# Module 5 — Inverse Kinematics
## Unit 4 — From Geometry to Numerical IK
### Lesson 4.3 — The Iteration Idea: Guess, Measure Error, Step

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
The guess-measure-step loop; the error history shrinks.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

def ik_numerical(target, theta0, L1, L2, alpha=0.5, tol=1e-4, max_iter=100):
    theta=np.array(theta0,float); target=np.array(target,float); hist=[]
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        hist.append(np.linalg.norm(e))
        if np.linalg.norm(e)<tol:
            return theta, it, hist
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        dtheta=np.linalg.solve(J,e)
        theta=theta+alpha*dtheta
    return theta, max_iter, hist

L1,L2=0.4,0.3
sol,iters,hist=ik_numerical([0.5,0.2],np.radians([10,20]),L1,L2)
print("converged in",iters,"iters to",np.round(np.degrees(sol),2),"deg")
print("error history:",[round(h,4) for h in hist[:6]],"...")


## Coding Exercise (§8)
Converge (FK-verify); decreasing error; two seeds → two solutions.


In [ ]:
# YOUR CODE HERE
sol,iters,hist=ik_numerical([0.5,0.2],np.radians([10,20]),L1,L2)
assert np.allclose(fk_two_link(sol[0],sol[1],L1,L2),[0.5,0.2],atol=1e-3)
assert all(hist[i+1]<=hist[i]+1e-9 for i in range(len(hist)-1))   # non-increasing
solA,_,_=ik_numerical([0.5,0.2],np.radians([10,20]),L1,L2)
solB,_,_=ik_numerical([0.5,0.2],np.radians([10,-20]),L1,L2)
# different seeds land on different elbow configs (theta2 opposite sign)
assert np.sign(solA[1])!=np.sign(solB[1])
print("numerical solver OK")


## Check your work


In [ ]:
sol,iters,hist=ik_numerical([0.5,0.2],np.radians([10,20]),L1,L2)
assert iters<100 and hist[-1]<1e-4
print("All checks passed.")
